In [152]:
import numpy as np
import pandas as pd
import joblib
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [153]:
np.random.seed(42)
n = 1000

data = pd.DataFrame({
    "idade": np.random.randint(20, 90, n),
    "frequencia_cardiaca": np.random.randint(60, 180, n),
    "spo2": np.random.randint(85, 100, n),
    "carga_sistema": np.random.uniform(0, 1, n),
    "recursos_disponiveis": np.random.randint(0, 10, n)
})

# Regra simulada de risco
data["pico_risco"] = (
    (data["frequencia_cardiaca"] > 140) |
    (data["spo2"] < 90) |
    (data["carga_sistema"] > 0.8)
).astype(int)

data.head()

,idade,frequencia_cardiaca,spo2,carga_sistema,recursos_disponiveis,pico_risco
0,71,133,99,0.739986,4,0
1,34,66,92,0.950313,2,1
2,80,92,99,0.202936,0,0
3,40,82,95,0.565550,0,0
4,43,144,86,0.979647,6,1


In [154]:
X = data.drop("pico_risco", axis=1)
y = data["pico_risco"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [155]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [156]:
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Acurácia: {acc:.4f}")
print("\nMatriz de Confusão:")
print(cm)
print("\nRelatório de Classificação:")
print(report)

Acurácia: 0.8250

Matriz de Confusão:
[[ 48  24]
 [ 11 117]]

Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.81      0.67      0.73        72
           1       0.83      0.91      0.87       128

    accuracy                           0.82       200
   macro avg       0.82      0.79      0.80       200
weighted avg       0.82      0.82      0.82       200



In [157]:
tn, fp, fn, tp = cm.ravel()

print(f"Verdadeiros Positivos (TP): {tp}")
print(f"Falsos Positivos (FP): {fp}")
print(f"Falsos Negativos (FN): {fn}")
print(f"Verdadeiros Negativos (TN): {tn}")

Verdadeiros Positivos (TP): 117
Falsos Positivos (FP): 24
Falsos Negativos (FN): 11
Verdadeiros Negativos (TN): 48


In [158]:
joblib.dump(model, "modelo_pico_risco.pkl")
print("Modelo salvo com sucesso!")

Modelo salvo com sucesso!


In [159]:
novo_paciente = pd.DataFrame([{
    "idade": 65,
    "frequencia_cardiaca": 150,
    "spo2": 88,
    "carga_sistema": 0.9,
    "recursos_disponiveis": 2
}])

prob = model.predict_proba(novo_paciente)[0][1]

print(f"Probabilidade de pico de risco: {prob:.2f}")

Probabilidade de pico de risco: 0.99


In [160]:
def classificar_risco(prob):
    if prob > 0.7:
        return "Alto risco"
    elif prob > 0.4:
        return "Médio risco"
    else:
        return "Baixo risco"

classe = classificar_risco(prob)

print(f"Classificação: {classe}")

Classificação: Alto risco


In [161]:
coeficientes = pd.DataFrame({
    "Feature": X.columns,
    "Peso": model.coef_[0]
}).sort_values(by="Peso", ascending=False)

coeficientes

,Feature,Peso
3,carga_sistema,2.954803
1,frequencia_cardiaca,0.043882
0,idade,0.000663
4,recursos_disponiveis,-0.000218
2,spo2,-0.355095


In [162]:
def prever_pico_risco(dados_paciente):
    df = pd.DataFrame([dados_paciente])
    prob = model.predict_proba(df)[0][1]

    if prob > 0.7:
        classe = "Alto risco"
    elif prob > 0.4:
        classe = "Médio risco"
    else:
        classe = "Baixo risco"

    return {
        "probabilidade": float(prob),
        "classificacao": classe
    }

# Teste
resultado = prever_pico_risco({
    "idade": 70,
    "frequencia_cardiaca": 160,
    "spo2": 87,
    "carga_sistema": 0.85,
    "recursos_disponiveis": 1
})

resultado

{'probabilidade': 0.9970591739984543, 'classificacao': 'Alto risco'}

In [163]:
def risco_tool(dados_paciente: dict):
    df = pd.DataFrame([dados_paciente])
    prob = model.predict_proba(df)[0][1]
    return float(prob)

In [164]:
class AgenteRisco:
    def executar(self, dados):
        prob = risco_tool(dados)
        return {
            "probabilidade": prob,
            "dados": dados
        }

In [165]:
class AgenteProtocolos:
    def executar(self, prob):
        if prob > 0.7:
            return {
                "risco": "Alto risco",
                "protocolo": "Monitoramento intensivo + suporte ventilatório + UTI"
            }
        elif prob > 0.4:
            return {
                "risco": "Médio risco",
                "protocolo": "Observação contínua + exames complementares"
            }
        else:
            return {
                "risco": "Baixo risco",
                "protocolo": "Monitoramento padrão"
            }

In [166]:
class Orquestrador:
    def __init__(self):
        self.agente_risco = AgenteRisco()
        self.agente_protocolos = AgenteProtocolos()

    def executar(self, dados_paciente):

        # HANDOFF 1
        resultado_risco = self.agente_risco.executar(dados_paciente)
        prob = resultado_risco["probabilidade"]

        # HANDOFF 2
        resultado_proto = self.agente_protocolos.executar(prob)

        # saída final estruturada
        return {
            "probabilidade": round(prob, 4),
            "classificacao": resultado_proto["risco"],
            "protocolo": resultado_proto["protocolo"]
        }

In [167]:
historico = []

In [168]:
orquestrador = Orquestrador()

paciente_teste = {
    "idade": 22,
    "frequencia_cardiaca": 70,
    "spo2": 90,
    "carga_sistema": 0.1,
    "recursos_disponiveis": 1
}

resultado = orquestrador.executar(paciente_teste)
resultado

{'probabilidade': 0.1921,
 'classificacao': 'Baixo risco',
 'protocolo': 'Monitoramento padrão'}

In [169]:


def log(agente, tipo, mensagem):
    historico.append({
        "agente": agente,
        "tipo": tipo,
        "mensagem": mensagem
    })

log("orquestrador", "entrada", paciente_teste)
log("analista_risco", "processamento", f"Probabilidade: {resultado['probabilidade']}")
log("protocolos", "saida", resultado)

import json

print(json.dumps(historico, indent=4, ensure_ascii=False))

[
    {
        "agente": "orquestrador",
        "tipo": "entrada",
        "mensagem": {
            "idade": 22,
            "frequencia_cardiaca": 70,
            "spo2": 90,
            "carga_sistema": 0.1,
            "recursos_disponiveis": 1
        }
    },
    {
        "agente": "analista_risco",
        "tipo": "processamento",
        "mensagem": "Probabilidade: 0.1921"
    },
    {
        "agente": "protocolos",
        "tipo": "saida",
        "mensagem": {
            "probabilidade": 0.1921,
            "classificacao": "Baixo risco",
            "protocolo": "Monitoramento padrão"
        }
    }
]
